<a href="https://colab.research.google.com/github/AnaraHayat/flyrank_assignment1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. Signal checks + my rule and its reason codes

### Signal A — staleness (behind FlyRank's refresh flags)
**Claim:** "Pages that haven't been updated in a long time (`days_since_last_update` high, `freshness_tier` = `91-180` or `181+`) are more likely to be declining (`is_declining_label`) than freshly-updated pages." This is the signal behind the refresh flag logic from the session.

### Signal B — volume (behind FlyRank's quick-win flag)
**Claim:** "Pages with more search impressions (`impressions_90d`, `impression_tier`) are more likely to be declining than low-impression pages" — the intuition being that a decline on a high-traffic page is the bigger, more worth-catching decline. This is the signal behind the quick-win flag.

Both signals are tested below against `is_declining_label = (trend_direction == "down")` — **the label itself is never a rule input**, only the target I'm checking the signals against.

### My rule, in plain words
"A page is worth flagging for refresh if it's **stale** (no update in 180+ days) **and** it still gets **meaningful search visibility** (≥ 500 impressions in the last 90 days). Among flagged pages, rank by how much visibility is at stake — the more impressions, the higher the score."

This only uses `days_since_last_update` and `impressions_90d` — both are knowable right now, neither is the label or a future window.

### Reason code + action label
- **Reason code:** `stale_visible_page` — fires when both `stale` and `visible` are true. Everything else gets `not_flagged`.
- **Action label:** `refresh` when the reason code is `stale_visible_page`, else `monitor`.

In [5]:
import os

REPO_URL = "https://github.com/AnaraHayat/flyrank_assignment1.git"
REPO_DIR = "/content/flyrank_assignment1"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    #print("Repo already present, pulling latest...")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
#print("cwd:", os.getcwd())


import numpy as np
import pandas as pd
from pathlib import Path

# Find the repo root by walking up from the current working directory
# until we find data/raw/content_refresh_anonymized.csv. Works no matter
# where the kernel's cwd starts (Colab, local Jupyter, etc.).
DATA_REL = "data/raw/content_refresh_anonymized.csv"

start = Path.cwd()
repo_root = None
for candidate in [start, *start.parents]:
    if (candidate / DATA_REL).exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        f"Couldn't find {DATA_REL} above {start}. "
        "Make sure you've cloned/opened the flyrank_assignment1 repo "
        "and this notebook is running from inside it."
    )

os.chdir(repo_root)

df = pd.read_csv(DATA_REL)

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

overall_rate = df["is_declining_label"].mean()
print(f"Overall decline rate: {overall_rate:.3f} (n={len(df)})")
print()

# --- Signal A: staleness (freshness_tier) vs decline rate ---
print("=== Signal A: freshness_tier vs decline rate ===")
staleness_table = df.groupby("freshness_tier").agg(
    n=("content_id", "size"),
    decline_rate=("is_declining_label", "mean"),
).sort_values("decline_rate")
print(staleness_table.round(3))
print()
staleness_spearman = df["days_since_last_update"].corr(df["is_declining_label"], method="spearman")
print(f"Spearman corr(days_since_last_update, is_declining_label): {staleness_spearman:.3f}")
print()

# --- Signal B: volume (impression_tier) vs decline rate ---
print("=== Signal B: impression_tier vs decline rate ===")
volume_table = df.groupby("impression_tier").agg(
    n=("content_id", "size"),
    decline_rate=("is_declining_label", "mean"),
    median_impressions=("impressions_90d", "median"),
)
# order tiers by their natural rank rather than alphabetically
tier_order = ["low", "moderate", "good", "excellent"]
volume_table = volume_table.reindex(tier_order)
print(volume_table.round(3))
print()
volume_spearman = df["impressions_90d"].corr(df["is_declining_label"], method="spearman")
print(f"Spearman corr(impressions_90d, is_declining_label): {volume_spearman:.3f}")

Already up to date.
Overall decline rate: 0.542 (n=30000)

=== Signal A: freshness_tier vs decline rate ===
                    n  decline_rate
freshness_tier                     
181+              174         0.471
0-30            20480         0.511
31-90             175         0.589
91-180           9171         0.611

Spearman corr(days_since_last_update, is_declining_label): 0.049

=== Signal B: impression_tier vs decline rate ===
                     n  decline_rate  median_impressions
impression_tier                                         
low              11248         0.454                31.0
moderate         10469         0.615               998.0
good              7205         0.586              7249.0
excellent         1078         0.462             48675.0

Spearman corr(impressions_90d, is_declining_label): 0.146


### Verdicts

- **Signal A (staleness) verdict: MIXED.** Every tier clears the **~*50-row sample-size floor (smallest is `31-90` at n=175), but the pattern isn't the clean "staler = more declining" story the refresh flag assumes. The most-stale tier (`181+`, n=174) has the *lowest* decline rate (0.471), while `91-180` (n=9,171) has the *highest* (0.611) — not monotonic. A quartile-based cut on the raw `days_since_last_update` and the Spearman correlation (~0.05) both confirm this: staleness alone barely moves the decline rate in this snapshot. Directionally, it is not a clean win for the refresh flag's core assumption on its own — worth pairing with another signal (which is exactly what my rule does).
- **Signal B (volume) verdict: MIXED, leaning CONFIRMED.** All four tiers clear the floor (smallest is `excellent`, n=1,078). The relationship isn't monotonic across the named tiers either (`low`→0.454, `moderate`→0.615, `good`→0.586, `excellent`→0.462 — an inverted-U, not a straight line), but the Spearman correlation (~0.146) is three times stronger than staleness's, and the bulk of the data (the `low`→`moderate` step, n=11,248 and n=10,469) does move the right direction. Volume carries more real signal here than staleness does, but it isn't a clean linear story either.

Neither signal alone is a strong, clean CONFIRMED — which is exactly why the rule below combines them (stale **and** visible) instead of leaning on either one in isolation.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

*Score = `stale * visible * log1p(impressions_90d)`, readable on purpose — a flagged page's rank is driven only by how much visibility is on the line. Unflagged pages score 0 and sort to the bottom.*

In [6]:
import numpy as np

STALE_DAYS = 180        # matches the 91-180 / 181+ freshness_tier cut used above
VISIBLE_IMPRESSIONS = 500  # matches the impression_tier "moderate"+ cut used above

df["stale"] = (df["days_since_last_update"] >= STALE_DAYS).astype(int)
df["visible"] = (df["impressions_90d"] >= VISIBLE_IMPRESSIONS).astype(int)

# Transparent score: no fitted weights, just the rule multiplied through.
# log1p(impressions) keeps the ranking readable while still ordering flagged pages
# by how much traffic is actually on the line.
df["baseline_score"] = df["stale"] * df["visible"] * np.log1p(df["impressions_90d"])

# One reason code, one action label -- straight from the same two flags.
df["reason_code"] = np.where(
    (df["stale"] == 1) & (df["visible"] == 1),
    "stale_visible_page",
    "not_flagged",
)
df["action_label"] = np.where(df["reason_code"] == "stale_visible_page", "refresh", "monitor")

df["rank"] = df["baseline_score"].rank(method="first", ascending=False).astype(int)

queue_cols = [
    "content_id", "client_id", "rank", "baseline_score", "reason_code", "action_label",
    "days_since_last_update", "impressions_90d", "freshness_tier", "impression_tier",
]
queue = df[queue_cols].sort_values("rank")

os.makedirs("work/outputs", exist_ok=True)
out_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(out_path, index=False)

print(f"Wrote {out_path}")
print(f"Rows: {len(queue)}")
print(f"Flagged (stale_visible_page): {(df['reason_code'] == 'stale_visible_page').sum()} "
      f"({(df['reason_code'] == 'stale_visible_page').mean():.1%})")
print()
print("Action label counts:")
print(queue["action_label"].value_counts())
print()
print("Top of the queue:")
print(queue.head(5))

Wrote work/outputs/baseline_action_score.csv
Rows: 30000
Flagged (stale_visible_page): 17 (0.1%)

Action label counts:
action_label
monitor    29983
refresh       17
Name: count, dtype: int64

Top of the queue:
                 content_id          client_id  rank  baseline_score  \
16751  content_cf56e2e2e282  client_7f2253d7e2     1       11.029699   
16514  content_7368877ea310  client_7f2253d7e2     2       10.993278   
7021   content_1bfaa38ff26c  client_7f2253d7e2     3       10.154869   
21268  content_0a91db491d14  client_7f2253d7e2     4        9.495519   
11489  content_5feee3994adb  client_7f2253d7e2     5        8.963544   

              reason_code action_label  days_since_last_update  \
16751  stale_visible_page      refresh                     194   
16514  stale_visible_page      refresh                     194   
7021   stale_visible_page      refresh                     194   
21268  stale_visible_page      refresh                     193   
11489  stale_visible_page 

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

*This week's card asks for the top 10 (the skeleton title says top-20 — 10 is what's required; a top-20 pass is optional and not done here). For each: the action, why it's there, and what would make it wrong.*

In [7]:
top10 = queue.head(10).merge(
    df[["content_id", "ctr", "avg_position", "trend_direction"]], on="content_id", how="left"
)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
print(top10[[
    "rank", "content_id", "client_id", "baseline_score", "reason_code", "action_label",
    "impressions_90d", "days_since_last_update", "avg_position", "trend_direction",
]].to_string(index=False))

 rank           content_id         client_id  baseline_score        reason_code action_label  impressions_90d  days_since_last_update  avg_position trend_direction
    1 content_cf56e2e2e282 client_7f2253d7e2       11.029699 stale_visible_page      refresh            61678                     194          19.7            down
    2 content_7368877ea310 client_7f2253d7e2       10.993278 stale_visible_page      refresh            59472                     194          24.8            down
    3 content_1bfaa38ff26c client_7f2253d7e2       10.154869 stale_visible_page      refresh            25715                     194          22.2            down
    4 content_0a91db491d14 client_7f2253d7e2        9.495519 stale_visible_page      refresh            13299                     193          10.5            down
    5 content_5feee3994adb client_7f2253d7e2        8.963544 stale_visible_page      refresh             7812                     194          39.0            down
    6 content_c2

### Top-10 commentary

All ten share the same reason code (`stale_visible_page`) and action (`refresh`) by construction — the score just orders them by how much visibility is on the line. One line each:

1. **#1** — `refresh`. Highest `impressions_90d` among stale pages, so the biggest audience is seeing a page that hasn't been touched in 180+ days. Would be wrong if this traffic is mostly branded/navigational query volume that a content refresh can't move.
2. **#2** — `refresh`. Same logic, next-highest visibility. Wrong if `avg_position` is already page-1 top-3 — refreshing a page that's already winning risks disturbing something that works.
3. **#3** — `refresh`. Large stale audience. Wrong if the staleness is cosmetic (e.g. a template footer edit bumped `days_since_last_update` without the content actually going out of date).
4. **#4** — `refresh`. Same pattern. Wrong if this client's whole catalog is stale (a client-wide CMS migration date), making "stale" a client artifact rather than a real content-age signal.
5. **#5** — `refresh`. Wrong if `content_type` is `feedly article` — those pages are aggregated content the team may not maintain the same way as owned keyword articles.
6. **#6** — `refresh`. Wrong if impressions are concentrated in a single spiky day rather than steady 90-day demand — the log1p(impressions) score can't see that shape.
7. **#7** — `refresh`. Wrong if `main_intent` is `navigational` — visitors already know where they're going, so a content refresh may not change outcomes much.
8. **#8** — `refresh`. Wrong if `avg_position == 0` (no position data) — the page could be getting impressions but effectively invisible, a different problem than "stale content."
9. **#9** — `refresh`. Wrong if `word_count` is unusually high already — the assumption behind "refresh" is often "thin/outdated," which doesn't apply to a long, thorough page.
10. **#10** — `refresh`. Wrong if this page's decline (if any) is `trend_direction == "up"` or `"flat"` — the rule doesn't check trend at all, so a stale-but-still-growing page would be flagged for the wrong reason.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

*Weak picks: any top-10 row where `trend_direction` isn't `down` is a real weak pick — the rule flags on staleness + visibility only, so it will happily flag pages that are stale but not actually declining. That's expected and worth checking, not a bug to silently patch.*

In [8]:
# --- Weak picks: how many of my top 10 are NOT actually declining? ---
not_declining_in_top10 = top10[top10["trend_direction"] != "down"]
print(f"Top-10 rows where trend_direction != 'down': {len(not_declining_in_top10)} / 10")
if len(not_declining_in_top10):
    print(not_declining_in_top10[["rank", "content_id", "trend_direction", "impressions_90d"]]
          .to_string(index=False))
print()

# how the rule does across the WHOLE flagged set, not just the top 10 (honesty check)
flagged = df[df["reason_code"] == "stale_visible_page"]
print(f"Decline rate among all {len(flagged)} flagged rows: {flagged['is_declining_label'].mean():.3f}")
print(f"Decline rate among all {len(df)} rows (base rate): {df['is_declining_label'].mean():.3f}")
print("-> the rule's lift over the base rate is the honest number to report, not top-10 alone.")
print()

# --- Leakage / label-derivation check ---
rule_inputs = {"days_since_last_update", "impressions_90d"}
label_derived_cols = {"trend_direction", "trend_pct", "is_declining_label",
                      "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
                      "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"}
product_flag_cols = set()  # none of FlyRank's own health_score / quick-win / needs-attention
                            # columns exist in this starter CSV -- nothing to exclude here,
                            # confirmed by checking docs/data-dictionary.md

leak = rule_inputs & label_derived_cols
print(f"Rule inputs: {sorted(rule_inputs)}")
print(f"Any rule input in the label-derived set?: {bool(leak)}")
assert not leak, "Leakage: a rule input is derived from the label / a future window."
print("No future-window or label-derived columns used as rule inputs. Confirmed clean.")
# --- write a small metrics receipt (worth committing, unlike the CSV) ---
import json as _json
metrics = {
    "rows_total": int(len(df)),
    "rows_flagged_stale_visible_page": int(len(flagged)),
    "flagged_share": float(len(flagged) / len(df)),
    "decline_rate_base": float(df["is_declining_label"].mean()),
    "decline_rate_flagged": float(flagged["is_declining_label"].mean()),
    "decline_rate_top10": float((top10["trend_direction"] == "down").mean()),
    "signal_staleness_spearman": float(staleness_spearman),
    "signal_staleness_verdict": "MIXED",
    "signal_volume_spearman": float(volume_spearman),
    "signal_volume_verdict": "MIXED_LEANING_CONFIRMED",
    "rule_thresholds": {"stale_days": STALE_DAYS, "visible_impressions": VISIBLE_IMPRESSIONS},
}
with open("work/outputs/baseline_metadata.json", "w") as f:
    _json.dump(metrics, f, indent=2)
print("Wrote work/outputs/baseline_metadata.json")
print(_json.dumps(metrics, indent=2))


Top-10 rows where trend_direction != 'down': 0 / 10

Decline rate among all 17 flagged rows: 0.941
Decline rate among all 30000 rows (base rate): 0.542
-> the rule's lift over the base rate is the honest number to report, not top-10 alone.

Rule inputs: ['days_since_last_update', 'impressions_90d']
Any rule input in the label-derived set?: False
No future-window or label-derived columns used as rule inputs. Confirmed clean.
Wrote work/outputs/baseline_metadata.json
{
  "rows_total": 30000,
  "rows_flagged_stale_visible_page": 17,
  "flagged_share": 0.0005666666666666667,
  "decline_rate_base": 0.5420666666666667,
  "decline_rate_flagged": 0.9411764705882353,
  "decline_rate_top10": 1.0,
  "signal_staleness_spearman": 0.049460324423171866,
  "signal_staleness_verdict": "MIXED",
  "signal_volume_spearman": 0.14576276815766165,
  "signal_volume_verdict": "MIXED_LEANING_CONFIRMED",
  "rule_thresholds": {
    "stale_days": 180,
    "visible_impressions": 500
  }
}


## Self-check

Before you submit, confirm each line honestly:

- [Done ] Every section above is filled — markdown thinking AND the code that backs it
- [Done ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [Done ] No client names, URLs, or private queries anywhere
- [Done] My claims use careful words: observed, measured, directional, decision-support
- [Done ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.